# Theorem 3.2: Non-Negative Covariance Between Hidden Units

This notebook follows the setup from `sample_data_viz.ipynb` and investigates Theorem 3.2 from Vladimirova et al., *Understanding Priors in Bayesian Neural Networks at the Unit Level* (arXiv:1810.05193).

The theorem states that, under the Gaussian-prior ReLU network assumptions, hidden units in the same layer have non-negative covariance. More strongly, for two hidden units in layer $\ell$,

$$\operatorname{Cov}
\left[
  \left(h^{(\ell)}\right)^s,
  \left(\tilde h^{(\ell)}\right)^t
\right] \ge 0, \quad s,t \in \mathbb{N}.$$

For the first hidden layer, the theorem predicts equality because distinct first-layer units are independent under the prior.

The WHest mini split gives us repeated draws of random ReLU MLP weights. We treat MLPs as prior samples, condition on a fixed Gaussian input, propagate that input through every network, and estimate same-layer unit covariances across the random-network axis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import whestbench as wb

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "legend.framealpha": 0.85,
    "lines.linewidth": 1.8,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def tag_activation(fig, kind):
    styles = {
        "pre": ("PRE-ACTIVATION (pre-ReLU)", "#6a1b9a"),
        "post": ("POST-ACTIVATION (post-ReLU)", "#1b5e20"),
        "moments": ("POWER COVARIANCES", "#37474f"),
    }
    label, color = styles[kind]
    fig.text(
        0.0, 1.0, label, ha="left", va="top", fontsize=8.5, fontweight="bold",
        color="white", zorder=1000,
        bbox=dict(boxstyle="round,pad=0.3", fc=color, ec="none", alpha=0.92),
    )

## 1. Load the same WHest mini split

The original visualization notebook uses the public mini split. We keep that exact dataset setup and only change the statistic being measured.

In [ ]:
REPO = "aicrowd/arc-whestbench-public-2026"
REVISION = "v1-phase1"
SPLIT = "mini"

ds = wb.load_dataset(REPO, revision=REVISION, split=SPLIT)
names = ds["mlp_name"]
all_layer_means = np.asarray(ds["all_layer_means"], dtype=np.float32)
final_means = np.asarray(ds["final_means"], dtype=np.float32)
n_mlps, depth, width = all_layer_means.shape

print(f"Loaded {n_mlps} MLPs from hf://{REPO}@{REVISION} [split={SPLIT}]")
print(f"depth={depth}  width={width}  all_layer_means={all_layer_means.shape}")
print(f"max|final_means - last layer| = {np.abs(final_means - all_layer_means[:, -1]).max():.3e}")

## 2. Turn the theorem into a WHest diagnostic

The theorem's covariance is over the random network prior, conditional on a fixed input. In this notebook:

1. Draw one fixed Gaussian input $x_0$ and rescale it to typical norm $\sqrt{d}$.
2. For each WHest MLP, run a deterministic forward pass through all 32 layers.
3. For each layer, form a matrix `H` with shape `(n_mlps, width)`, where rows are prior draws and columns are hidden units.
4. Estimate off-diagonal covariances between same-layer unit columns.

This directly targets the theorem's prior covariance. It is different from covariance across input samples inside one fixed MLP, which appears elsewhere in the visualization notebook.

In [ ]:
rng = np.random.default_rng(320181005193)
x0 = rng.standard_normal(width).astype(np.float64)
x0 *= np.sqrt(width) / np.linalg.norm(x0)

pre_units = np.empty((n_mlps, depth, width), dtype=np.float32)
post_units = np.empty((n_mlps, depth, width), dtype=np.float32)

for mlp_idx in range(n_mlps):
    h = x0.copy()
    for layer_idx, W in enumerate(wb.mlp_at(ds, mlp_idx).weights):
        g = h @ np.asarray(W, dtype=np.float64)
        h = np.maximum(g, 0.0)
        pre_units[mlp_idx, layer_idx] = g.astype(np.float32)
        post_units[mlp_idx, layer_idx] = h.astype(np.float32)

print(f"fixed input norm: {np.linalg.norm(x0):.3f} (target sqrt(width) = {np.sqrt(width):.3f})")
print(f"collected pre_units={pre_units.shape}, post_units={post_units.shape}")
print(f"prior samples per unit column: {n_mlps}; unit columns per layer: {width}")

## 3. Pairwise covariance summary

For each layer, we compute the off-diagonal covariance matrix across unit columns. The theorem predicts non-negative population covariance. With only 100 network draws, individual pair estimates can be noisy, so the most stable diagnostic is the distribution and mean of off-diagonal covariances rather than requiring every finite-sample pair to be positive.

In [ ]:
def offdiag_values(matrix):
    mask = ~np.eye(matrix.shape[0], dtype=bool)
    return matrix[mask]


def cross_cov_columns(A, B=None):
    A = np.asarray(A, dtype=np.float64)
    B = A if B is None else np.asarray(B, dtype=np.float64)
    A = A - A.mean(axis=0, keepdims=True)
    B = B - B.mean(axis=0, keepdims=True)
    return A.T @ B / (A.shape[0] - 1)


def covariance_summary(units):
    mean_cov = np.empty(depth)
    median_cov = np.empty(depth)
    q10_cov = np.empty(depth)
    q90_cov = np.empty(depth)
    negative_frac = np.empty(depth)
    for layer_idx in range(depth):
        cov = cross_cov_columns(units[:, layer_idx, :])
        offdiag = offdiag_values(cov)
        mean_cov[layer_idx] = offdiag.mean()
        median_cov[layer_idx] = np.median(offdiag)
        q10_cov[layer_idx], q90_cov[layer_idx] = np.quantile(offdiag, [0.10, 0.90])
        negative_frac[layer_idx] = (offdiag < 0).mean()
    return mean_cov, median_cov, q10_cov, q90_cov, negative_frac


pre_mean_cov, pre_med_cov, pre_q10_cov, pre_q90_cov, pre_neg_frac = covariance_summary(pre_units)
post_mean_cov, post_med_cov, post_q10_cov, post_q90_cov, post_neg_frac = covariance_summary(post_units)

print("layer  pre mean cov  post mean cov  post negative-pair fraction")
for layer_idx in [0, 1, 2, 3, 7, 15, 31]:
    print(f"{layer_idx + 1:>5}  {pre_mean_cov[layer_idx]:>12.3e}  {post_mean_cov[layer_idx]:>13.3e}  {post_neg_frac[layer_idx]:>25.1%}")

In [ ]:
x_layers = np.arange(1, depth + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].fill_between(x_layers, post_q10_cov, post_q90_cov, color="#1b5e20", alpha=0.18, label="post-ReLU 10-90% pair band")
axes[0].plot(x_layers, post_mean_cov, "o-", color="#1b5e20", ms=4, label="post-ReLU mean off-diag covariance")
axes[0].plot(x_layers, pre_mean_cov, "s-", color="#6a1b9a", ms=4, label="pre-ReLU mean off-diag covariance")
axes[0].axhline(0, color="black", lw=1.0, ls="--")
axes[0].set(title="Mean same-layer covariance across random MLPs", xlabel="layer ell", ylabel="off-diagonal covariance")
axes[0].legend(fontsize=8)

axes[1].plot(x_layers, post_neg_frac, "o-", color="crimson", ms=4, label="post-ReLU negative pair fraction")
axes[1].plot(x_layers, pre_neg_frac, "s-", color="#6a1b9a", ms=4, label="pre-ReLU negative pair fraction")
axes[1].axhline(0.5, color="gray", lw=1.0, ls=":", label="noise-like 50% reference")
axes[1].set(title="Finite-sample pair signs", xlabel="layer ell", ylabel="fraction of off-diagonal pairs < 0", ylim=(0, 1))
axes[1].legend(fontsize=8)

plt.tight_layout()
tag_activation(fig, "post")
plt.show()

## 4. The stronger power-covariance claim

The theorem also asserts non-negative covariance between powers of same-layer hidden units. We check a small grid of powers, using post-ReLU units because the theorem statement uses $h^{(\ell)}$. Each entry below is the mean off-diagonal covariance

$$\operatorname{Cov}
\left[
  \left(h_i^{(\ell)}\right)^s,
  \left(h_j^{(\ell)}\right)^t
\right], \quad i \ne j,$$

estimated across the 100 MLP draws.

In [ ]:
POWER_PAIRS = [(1, 1), (1, 2), (2, 1), (2, 2), (1, 3), (3, 1)]
power_mean_cov = np.empty((len(POWER_PAIRS), depth), dtype=np.float64)
power_neg_frac = np.empty((len(POWER_PAIRS), depth), dtype=np.float64)

for pair_idx, (s, t) in enumerate(POWER_PAIRS):
    for layer_idx in range(depth):
        H = post_units[:, layer_idx, :].astype(np.float64)
        cov = cross_cov_columns(H**s, H**t)
        offdiag = offdiag_values(cov)
        power_mean_cov[pair_idx, layer_idx] = offdiag.mean()
        power_neg_frac[pair_idx, layer_idx] = (offdiag < 0).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for pair_idx, (s, t) in enumerate(POWER_PAIRS):
    axes[0].plot(x_layers, power_mean_cov[pair_idx], marker="o", ms=3, lw=1.3, label=f"s={s}, t={t}")
axes[0].axhline(0, color="black", lw=1.0, ls="--")
axes[0].set(title="Mean off-diagonal power covariance", xlabel="layer ell", ylabel="mean covariance")
axes[0].legend(ncol=2, fontsize=8)

for pair_idx, (s, t) in enumerate(POWER_PAIRS):
    axes[1].plot(x_layers, power_neg_frac[pair_idx], marker="s", ms=3, lw=1.3, label=f"s={s}, t={t}")
axes[1].set(title="Finite-sample negative pair fraction for powers", xlabel="layer ell", ylabel="fraction < 0", ylim=(0, 1))
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()
tag_activation(fig, "moments")
plt.show()

## 5. Layer snapshots: covariance matrix structure

A layer-level mean can hide structure. These heatmaps show the off-diagonal covariance matrix for selected layers after normalizing each hidden unit to unit variance across MLP draws, i.e. an empirical correlation matrix over the prior.

In [ ]:
snapshot_layers = [0, 1, 7, 15, 31]
fig, axes = plt.subplots(1, len(snapshot_layers), figsize=(16, 3.5), constrained_layout=True)

for ax, layer_idx in zip(axes, snapshot_layers):
    H = post_units[:, layer_idx, :].astype(np.float64)
    H = H - H.mean(axis=0, keepdims=True)
    scale = H.std(axis=0, ddof=1, keepdims=True)
    H = H / np.maximum(scale, 1e-12)
    corr = H.T @ H / (H.shape[0] - 1)
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
    ax.set_title(f"ell={layer_idx + 1}")
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.78, label="empirical correlation")
tag_activation(fig, "post")
plt.show()

## 6. Robustness across several fixed inputs

The theorem is conditional on the input. A single $x_0$ is enough for the theoretical statement, but a few independent fixed inputs help reveal whether the empirical sign pattern is stable or an accident of one input vector.

In [ ]:
N_INPUT_CHECKS = 4
robust_post_mean_cov = np.empty((N_INPUT_CHECKS, depth), dtype=np.float64)

for input_idx in range(N_INPUT_CHECKS):
    x = rng.standard_normal(width).astype(np.float64)
    x *= np.sqrt(width) / np.linalg.norm(x)
    post_tmp = np.empty((n_mlps, depth, width), dtype=np.float32)
    for mlp_idx in range(n_mlps):
        h = x.copy()
        for layer_idx, W in enumerate(wb.mlp_at(ds, mlp_idx).weights):
            g = h @ np.asarray(W, dtype=np.float64)
            h = np.maximum(g, 0.0)
            post_tmp[mlp_idx, layer_idx] = h.astype(np.float32)
    robust_post_mean_cov[input_idx] = covariance_summary(post_tmp)[0]
    print(f"finished fixed input {input_idx + 1}/{N_INPUT_CHECKS}")

lo, mid, hi = np.percentile(robust_post_mean_cov, [10, 50, 90], axis=0)
fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(x_layers, lo, hi, color="#1b5e20", alpha=0.18, label="10-90% band over fixed inputs")
ax.plot(x_layers, mid, "o-", color="#1b5e20", ms=4, label="median over fixed inputs")
ax.axhline(0, color="black", lw=1.0, ls="--")
ax.set(title="Post-ReLU mean covariance across fixed inputs", xlabel="layer ell", ylabel="mean off-diagonal covariance")
ax.legend(fontsize=8)
plt.tight_layout()
tag_activation(fig, "post")
plt.show()

## 7. WHest target-level variant: covariance of activation means

The stored `all_layer_means` are post-ReLU expectations over the Gaussian input distribution for each fixed MLP. That is not the exact conditional-on-one-input object in Theorem 3.2, but it is directly relevant to WHest because estimators predict these means. This cell asks whether the non-negative covariance pattern is also visible after input averaging.

In [ ]:
target_mean_cov, target_med_cov, target_q10_cov, target_q90_cov, target_neg_frac = covariance_summary(all_layer_means)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].fill_between(x_layers, target_q10_cov, target_q90_cov, color="teal", alpha=0.18, label="10-90% pair band")
axes[0].plot(x_layers, target_mean_cov, "o-", color="teal", ms=4, label="mean off-diag covariance")
axes[0].axhline(0, color="black", lw=1.0, ls="--")
axes[0].set(title="Covariance of WHest activation means across MLPs", xlabel="layer ell", ylabel="off-diagonal covariance")
axes[0].legend(fontsize=8)

axes[1].plot(x_layers, target_neg_frac, "o-", color="crimson", ms=4)
axes[1].set(title="Negative pair fraction after input averaging", xlabel="layer ell", ylabel="fraction of pairs < 0", ylim=(0, 1))

plt.tight_layout()
tag_activation(fig, "post")
plt.show()

## 8. Interpretation

Read these diagnostics as finite-sample checks of the theorem's sign claim, not as a proof. The population statement is non-negative covariance across the Gaussian weight prior. The mini split has only 100 MLP draws, so individual unit-pair estimates can be negative from sampling noise even when the mean off-diagonal covariance is positive.

The cleanest expected pattern is:

- layer 1 covariance is near zero, matching the equality case;
- deeper-layer mean off-diagonal covariance becomes positive;
- power covariances for small $s,t$ are also positive on average;
- the WHest target-level variant often shows the same dependence structure after averaging over inputs, which helps explain why final-layer activation means are statistically coupled rather than independent coordinate-wise targets.